# 02 Query、Key、Value 是什么

上一节我们已经把 Attention 的最小流程讲清楚了：

```text
相关性分数 -> Softmax -> 注意力权重 -> 加权汇总
```

但是上一节故意留下了一个问题：

```text
相关性分数到底怎么计算？
```

这一节学习 Query、Key、Value，也就是 $Q$、$K$、$V$。

它们不是三个孤立名词，也不是三份互不相关的数据。

$Q$、$K$、$V$ 的作用是把 Attention 里面的三件事说清楚：

```text
我现在想找什么？
每份信息拿什么来和我匹配？
最后真正汇总哪部分内容？
```

## 1. 为什么这一节要学 QKV

上一节说 Attention 是动态加权汇总。

也就是说，模型不是平均看所有信息，而是先判断谁更相关，再给更相关的信息更大的权重。

现在最关键的问题变成：

```text
模型怎么判断“谁和谁更相关”？
```

如果没有 QKV，我们只能很笼统地说：模型计算相关性分数。

但这句话还不够清楚。

QKV 就是用来把“相关性分数怎么来”这件事拆开的。

## 2. 先把 Attention 里的角色分清楚

在一次 Attention 计算里，至少有两个角色。

第一个角色：当前需求。

也就是现在这个位置想从其他信息里找到什么。

第二个角色：候选信息。

也就是输入里可以被参考、被汇总的那些信息。

举个简单句子：

```text
小明把复习资料交给小红，因为她明天要考试。
```

如果当前要理解“她”，那么“她”这个位置就像当前需求。

句子里的其他词，比如“小明”“复习资料”“小红”“考试”，就是候选信息。

Attention 要做的是：

```text
让当前需求分别和每个候选信息比较，看看谁更相关。
```

## 3. $Q$、$K$、$V$ 分别表示什么

$Q$、$K$、$V$ 的全称分别是：

$$
\begin{aligned}
Q &= \\mathrm{Query} \\
K &= \\mathrm{Key} \\
V &= \\mathrm{Value}
\end{aligned}
$$

可以先这样理解：

```text
Query：为了更好地理解或更新当前这个词，我需要从其他词那里寻找什么信息？
Key：我具有什么可供匹配的特征
Value：如果你选择关注我，我真正提供什么内容？
```

换成更接近计算过程的话：

- $Q$ 和 $K$ 用来计算相关性分数。
- $V$ 用来做最后的加权求和。

所以 QKV 的分工非常明确：

- $Q$：提出匹配需求。
- $K$：提供匹配依据。
- $V$：提供汇总内容。

## 4. 用查资料来做一个直观类比

为了先建立直觉，可以把 Attention 想成一次查资料。

假设你现在想找“谁明天要考试”。

这个查找需求就是 Query。

每份资料都有一些标签或目录信息，比如“人物”“动作”“时间”“原因”。这些用来和需求匹配的信息，可以类比成 Key。

真正被你读进去、拿来回答问题的正文内容，可以类比成 Value。

过程就是：

```text
用 Query 去匹配每份资料的 Key。
匹配程度越高，权重越大。
最后按权重阅读并汇总 Value。
```

这个类比只是帮助理解分工。

真正的模型里，$Q$、$K$、$V$ 都是向量，不是人工写好的文字标签。

## 5. $Q$ 和 $K$ 怎么计算相关性分数

现在假设有一个 Query 向量 $\mathbf{q}$，还有三份候选信息的 Key 向量：

$$
\mathbf{k}_1,\mathbf{k}_2,\mathbf{k}_3
$$

模型需要判断 Query 和每个 Key 有多匹配。

最常见的一种做法是点积：

$$
s_i=\mathbf{q}\cdot\mathbf{k}_i
$$

这里：

- $s_i$ 是第 $i$ 份信息的相关性分数。
- $\mathbf{q}$ 表示当前需求。
- $\mathbf{k}_i$ 表示第 $i$ 份信息的匹配依据。

点积越大，可以先理解为两个向量越匹配。

于是就得到了上一节说的相关性分数。

## 6. 从分数到权重：Softmax 继续登场

假设 $Q$ 和三个 $K$ 算出来的相关性分数是：

$$
\begin{aligned}
\mathrm{score} &= [1.0, 3.0, 2.0]
\end{aligned}
$$

这些分数还不是权重。

因为它们可能是负数，也不一定加起来等于 1。

所以要经过 Softmax：

$$
\alpha_i=\frac{\exp(s_i)}{\sum_j\exp(s_j)}
$$

得到注意力权重：

$$
\begin{aligned}
\alpha &= [0.090, 0.665, 0.245]
\end{aligned}
$$

这里就能看出 $Q$、$K$ 的作用了：

- $Q$ 和 $K$ 不直接输出最终结果。
- 它们负责决定每份信息应该占多大权重。

## 7. $V$ 才是真正被加权汇总的内容

得到注意力权重之后，模型要汇总信息。

这时候参与汇总的不是 Key，而是 Value。

假设三份信息对应的 Value 是：

$$
\mathbf{v}_1,\mathbf{v}_2,\mathbf{v}_3
$$

加权汇总就是：

$$
\mathbf{c}=\alpha_1\mathbf{v}_1+\alpha_2\mathbf{v}_2+\alpha_3\mathbf{v}_3
$$

也可以写成求和形式：

$$
\mathbf{c}=\sum_i\alpha_i\mathbf{v}_i
$$

这里一定要分清楚：

```text
Key 用来匹配。
Value 用来汇总。
```

如果把 $K$ 和 $V$ 混成一个概念，后面学 Self-Attention 和 Multi-Head Attention 会很容易乱。

## 8. 用一个小数字例子完整走一遍

假设当前 Query 是：

$$
\mathbf{q}=[1,0]
$$

三个候选信息的 Key 是：

$$
\mathbf{k}_1=[1,0]
$$

$$
\mathbf{k}_2=[0,1]
$$

$$
\mathbf{k}_3=[0.8,0.2]
$$

先算点积分数：

$$
s_1=[1,0]\cdot[1,0]=1
$$

$$
s_2=[1,0]\cdot[0,1]=0
$$

$$
s_3=[1,0]\cdot[0.8,0.2]=0.8
$$

所以：

$$
\begin{aligned}
\mathrm{score} &= [1.0, 0.0, 0.8]
\end{aligned}
$$

经过 Softmax 后，权重大约是：

$$
\begin{aligned}
\alpha &= [0.457, 0.168, 0.375]
\end{aligned}
$$

如果三个 Value 是：

$$
\mathbf{v}_1=[10,0]
$$

$$
\mathbf{v}_2=[0,10]
$$

$$
\mathbf{v}_3=[8,2]
$$

最后输出就是：

$$
\mathbf{c}=0.457\mathbf{v}_1+0.168\mathbf{v}_2+0.375\mathbf{v}_3
$$

这个结果会更接近 $\mathbf{v}_1$ 和 $\mathbf{v}_3$，因为它们的 Key 和 Query 更匹配。

这就是一次完整的 QKV Attention。

## 9. $Q$、$K$、$V$ 是从哪里来的

现在再回答一个非常重要的问题：

$Q$、$K$、$V$ 是不是原始输入里本来就有的三份数据？

不是。

在常见的神经网络里，输入先表示成一组向量，比如：

$$
\mathbf{x}_1,\mathbf{x}_2,\dots,\mathbf{x}_N
$$

然后模型用三组不同的线性变换，把同一个输入表示变成 $Q$、$K$、$V$：

$$
\mathbf{q}_i=\mathbf{x}_i\mathbf{W}_Q
$$

$$
\mathbf{k}_i=\mathbf{x}_i\mathbf{W}_K
$$

$$
\mathbf{v}_i=\mathbf{x}_i\mathbf{W}_V
$$

这里 $\mathbf{W}_Q$、$\mathbf{W}_K$、$\mathbf{W}_V$ 是模型要学习的参数。

所以要记住：

$Q$、$K$、$V$ 通常来自同一份输入，只是通过不同的可学习变换得到。

## 10. 为什么要分成 $Q$、$K$、$V$ 三套表示

如果都是从同一个输入来的，为什么还要分成三份？

先不要从公式理解，先从“同一个词有不同用途”理解。

一句话里的每个词，本来都有一个输入表示，比如第 $i$ 个词是：

$$
\mathbf{x}_i
$$

但是在 Attention 里，这个词可能同时扮演三种角色：

```text
当它主动去找别的信息时，它需要 Query 表示。
当它被别的位置拿来匹配时，它需要 Key 表示。
当它的内容被汇总进结果时，它需要 Value 表示。
```

所以 $Q$、$K$、$V$ 不是三份毫无关系的数据。

更准确地说：

- 同一个输入 $x_{i}$
- 经过 $W_{Q}$ 变成 $q_{i}$：用来提问
- 经过 $W_{K}$ 变成 $k_{i}$：用来被匹配
- 经过 $W_{V}$ 变成 $v_{i}$：用来提供内容

可以把 $\mathbf{W}_Q$、$\mathbf{W}_K$、$\mathbf{W}_V$ 想成三副不同的眼镜。

同一个词戴上不同眼镜，会被看成不同用途的表示。

这样模型就可以更灵活地学习：

```text
什么特征适合拿来提问？
什么特征适合拿来匹配？
什么特征适合拿来汇总？
```

## 11. 先别急着看 $B \times N \times D$：从一个位置开始

你可能从这里开始觉得乱，主要是因为一下子出现了太多维度。

我们先只看一句话里的一个位置。

假设这个位置原来的向量是：

$$
\mathbf{x}_i
$$

它经过三次变换，得到三种表示：

$$
\mathbf{q}_i=\mathbf{x}_i\mathbf{W}_Q
$$

$$
\mathbf{k}_i=\mathbf{x}_i\mathbf{W}_K
$$

$$
\mathbf{v}_i=\mathbf{x}_i\mathbf{W}_V
$$

先只记住一句话：

一个输入位置 $x_{i}$，会生成一个 $q_{i}$、一个 $k_{i}$、一个 $v_{i}$。

这一步还没有和别的位置发生关系，只是在给这个位置准备三种身份。

## 12. 再看一句话：$N$ 个位置就有 $N$ 套 QKV

如果一句话有 $N$ 个位置：

$$
\mathbf{x}_1,\mathbf{x}_2,\dots,\mathbf{x}_N
$$

那么每个位置都会生成自己的 $Q$、$K$、$V$：

- $x_{1}$ -> $q_{1}$, $k_{1}$, $v_{1}$
- $x_{2}$ -> $q_{2}$, $k_{2}$, $v_{2}$
- ...
- $x_{N}$ -> $q_{N}$, $k_{N}$, $v_{N}$

把所有 Query 放在一起，就叫 $Q$。

把所有 Key 放在一起，就叫 $K$。

把所有 Value 放在一起，就叫 $V$。

所以这里的大写 $Q$、$K$、$V$ 可以先理解成三张表：

- $Q$ 表：每个位置的“提问表示”。
- $K$ 表：每个位置的“被匹配表示”。
- $V$ 表：每个位置的“内容表示”。

## 13. 用形状把 QKV 串起来

假设输入形状是：

$$
B\times N\times D
$$

其中：

- $B$ 表示一批有多少个样本。
- $N$ 表示每个样本里有多少个位置。
- $D$ 表示每个位置的特征维度。

现在把它翻译成人话：

- $B$：一次看多少条句子或样本。
- $N$：每条句子里有多少个位置。
- $D$：每个位置原来用多少个数字表示。

举个小例子：

$$
\begin{aligned}
B &= 2 \quad \text{一批里有 2 句话} \\
N &= 4 \quad \text{每句话先假设有 4 个词} \\
D &= 8 \quad \text{每个词用 8 个数字表示}
\end{aligned}
$$

那输入就是：

$$
\begin{aligned}
2 \times 4 \times 8
\end{aligned}
$$

经过三组线性变换后，位置数量不会变，batch 数量也不会变。

变的是每个位置的表示方式：

$$
\begin{aligned}
Q &: B \times N \times d_{k} \\
K &: B \times N \times d_{k} \\
V &: B \times N \times d_{v}
\end{aligned}
$$

这里最重要的是：

- $B$ 和 $N$ 保持不变。
- 每个位置只是从原来的 $x$ 表示，变成了 $q$、$k$、$v$ 三种表示。

为什么 $Q$ 和 $K$ 的最后一维通常一样？

因为它们要做点积匹配。

点积要求两个向量维度一致。

Value 的维度可以记作 $d_v$，因为它是最后被汇总的内容维度。

这一节先不展开矩阵乘法细节，只先记住形状上的角色：

- $Q$ 和 $K$ 的维度要能匹配。
- $V$ 的维度决定最后输出内容的维度。

## 14. 一个 Query 怎么看所有 Key

现在开始发生“注意力”。

假设我们正在处理第 2 个位置。

第 2 个位置会拿自己的 Query：

$$
\mathbf{q}_2
$$

去和所有位置的 Key 做匹配：

$$
\mathbf{k}_1,\mathbf{k}_2,\dots,\mathbf{k}_N
$$

匹配方式可以先理解成点积：

$$
s_{2,j}=\mathbf{q}_2\cdot\mathbf{k}_j
$$

这个符号读作：

```text
第 2 个位置，看第 j 个位置时的相关性分数。
```

如果有 4 个位置，就会得到 4 个分数：

- $q_{2}$ 看 $k_{1}$ 的分数
- $q_{2}$ 看 $k_{2}$ 的分数
- $q_{2}$ 看 $k_{3}$ 的分数
- $q_{2}$ 看 $k_{4}$ 的分数

这 4 个分数表示：第 2 个位置分别应该多关注其他每个位置。

## 15. 一个 Query 的完整三步

现在把第 2 个位置的 Attention 完整走一遍。

第一步，计算分数：

$$
s_{2,j}=\mathbf{q}_2\cdot\mathbf{k}_j
$$

这一步得到的是第 2 个位置看所有位置的原始分数。

第二步，对这些分数做 Softmax，得到权重：

$$
\alpha_{2,j}=\operatorname{softmax}(s_{2,j})
$$

这一步得到的是第 2 个位置看所有位置的注意力权重。

第三步，用这些权重对所有 Value 加权求和：

$$
\mathbf{c}_2=\sum_j\alpha_{2,j}\mathbf{v}_j
$$

这里的 $\mathbf{c}_2$ 就是第 2 个位置更新后的新表示。

一句话概括：

第 2 个位置先用 $q_{2}$ 去匹配所有 $k$，得到权重，最后按权重汇总所有 $v$。

别急着管矩阵公式，先把这件事想明白。

## 16. 为什么会得到 $N \times N$ 的分数表

上面我们只看了第 2 个位置。

但真实情况下，每个位置都要做同样的事情。

如果有 $N$ 个位置，就有 $N$ 个 Query：

$$
\mathbf{q}_1,\mathbf{q}_2,\dots,\mathbf{q}_N
$$

也有 $N$ 个 Key：

$$
\mathbf{k}_1,\mathbf{k}_2,\dots,\mathbf{k}_N
$$

每个 Query 都要去看所有 Key。

所以会得到一张分数表：

$$
\begin{array}{c|ccccc}
 & k_1 & k_2 & k_3 & \cdots & k_N \\
q_1 & s_{11} & s_{12} & s_{13} & \cdots & s_{1N} \\
q_2 & s_{21} & s_{22} & s_{23} & \cdots & s_{2N} \\
q_3 & s_{31} & s_{32} & s_{33} & \cdots & s_{3N} \\
\vdots & \vdots & \vdots & \vdots & \ddots & \vdots \\
q_N & s_{N1} & s_{N2} & s_{N3} & \cdots & s_{NN}
\end{array}
$$

这就是 $N\times N$ 的来源。

行表示 Query，也就是“谁在看”。

列表示 Key，也就是“被看的是谁”。

每一行做一次 Softmax，就得到这个 Query 对所有位置的注意力权重。

## 17. Value 怎么被汇总

有了注意力权重之后，才轮到 Value 出场。

还是看第 2 个位置。

第 2 个位置得到一行权重：

$\alpha_{21}$, $\alpha_{22}$, $\alpha_{23}$, ..., $\alpha_{2N}$

然后它用这些权重去加权所有 Value：

$$
\mathbf{c}_2=\alpha_{2,1}\mathbf{v}_1+\alpha_{2,2}\mathbf{v}_2+\dots+\alpha_{2,N}\mathbf{v}_N
$$

注意，这里汇总的是所有 Value，不是所有 Key。

所以一定要记住：

- $K$ 决定匹配分数。
- $V$ 提供最终被拿走的内容。

每个 Query 都会得到一个新的输出表示。

所以 $N$ 个 Query 最后会得到 $N$ 个新表示。

## 18. 再看矩阵公式：它只是批量写法

后面你会经常看到这个公式：

$$
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V
$$

第一次看到它很正常会懵。

但它不是新东西，只是把前面一行一行的计算合在一起写。

但可以先知道里面每一部分的大意：

- $QK^{\top}$：让每个 Query 都和每个 Key 算分数，得到 $N \times N$ 分数表。
- $\sqrt{d_{k}}$：对分数做缩放，避免数值过大。
- softmax：对每一行分数做归一化，得到每个 Query 的注意力权重。
- 最后乘 $V$：每一行权重都去加权汇总所有 Value。

所以这个公式可以用一句话读：

```text
先算所有位置之间的匹配分数，再变成权重，最后按权重汇总内容。
```

## 19. 为什么公式里经常有除以根号 $d_{k}$

这一小节先当作补充理解，不要把它当成主线。

$Q$ 和 $K$ 做点积时，如果向量维度 $d_k$ 很大，点积结果可能也会偏大。

分数太大时，Softmax 很容易变得特别极端：

```text
一个位置权重接近 1
其他位置权重接近 0
```

这会让训练不太稳定。

所以常见做法是把分数除以：

$$
\sqrt{d_k}
$$

它的作用可以先理解成：

```text
把点积分数缩小一点，让 Softmax 不要太激进。
```

缩放之后，分数更稳定，训练也更容易。

这一节只需要知道它是为了稳定分数，不需要推导统计细节。

## 20. 不要把 QKV 理解错

这一节最容易出现几个误解。

第一个误解：$Q$、$K$、$V$ 是三份完全不同的原始数据。

不对。很多时候它们来自同一份输入，只是经过不同线性变换。

第二个误解：$K$ 和 $V$ 是一回事。

不对。$K$ 用来匹配，$V$ 用来汇总。

第三个误解：注意力权重是训练后固定保存下来的参数。

不对。注意力权重是根据当前输入临时算出来的。真正训练学习的是生成 $Q$、$K$、$V$ 等表示时用到的参数。

第四个误解：学 QKV 就等于学完 Transformer。

不对。QKV 只是 Attention 的核心部件。后面还要继续学习 Self-Attention、Multi-Head Attention、位置编码和 Transformer Encoder。

## 21. 本节小结

这一节先记住一条主线：

- $Q$ 提出需求。
- $K$ 提供匹配依据。
- $Q$ 和 $K$ 计算相关性分数。
- Softmax 把分数变成注意力权重。
- $V$ 提供真正被汇总的内容。
- 权重作用在 $V$ 上，得到最终输出。

核心公式可以先记成两层。

第一层：单个 Query 看多个 Key 和 Value。

$$
s_{i,j}=\mathbf{q}_i\cdot\mathbf{k}_j
$$

$$
\alpha_{i,j}=\operatorname{softmax}(s_{i,j})
$$

$$
\mathbf{c}_i=\sum_j\alpha_{i,j}\mathbf{v}_j
$$

第二层：所有 Query 一起算，就是矩阵公式。

$$
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V
$$

更口语一点说：

```text
拿需求去匹配标签，再按匹配程度汇总内容。
```

## 22. 自测问题

1. 为什么学 QKV 前，要先理解“相关性分数从哪里来”？
2. Query、Key、Value 的全称分别是什么？
3. Query 可以先理解成什么？
4. Key 在 Attention 中负责什么？
5. Value 在 Attention 中负责什么？
6. 为什么说 $Q$ 和 $K$ 用来算分数，$V$ 用来做汇总？
7. $Q$、$K$、$V$ 通常是不是三份完全不同的原始数据？为什么？
8. $\mathbf{W}_Q$、$\mathbf{W}_K$、$\mathbf{W}_V$ 是什么？
9. 一个输入位置 $\mathbf{x}_i$ 会生成哪三个表示？
10. 为什么 $Q$ 和 $K$ 的最后一维通常要一样？
11. $s_{i,j}$ 表示什么意思？
12. 多个 Query 和多个 Key 匹配时，为什么会得到 $N\times N$ 的分数表？
13. 公式里的 $QK^{\top}$ 表示什么？
14. 为什么最后要乘 $V$？
15. 为什么常见 Attention 公式里要除以 $\sqrt{d_k}$？